In [1]:
import json
from pymongo import MongoClient
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from joblib import Parallel, delayed
import nltk

# Télécharger les corpus nécessaires pour NLTK
nltk.download('punkt')

# Connexion à MongoDB
def get_data_from_mongodb():
    client = MongoClient('mongodb://localhost:27017')
    db = client['dataset']
    collection = db['LesArticles']
    documents = list(collection.find({
        'cluster_id': {'$exists': True},
        'content': {'$exists': True, '$ne': ''}
    }))
    return db, documents

# Fonction pour générer un résumé avec LSA
def generate_summary(text, sentences_count=4):
    try:
        if not text or not isinstance(text, str):
            return "Texte vide ou invalide"
        
        # Limiter le texte pour accélérer le traitement
        text = text[:4000]

        parser = PlaintextParser.from_string(text, Tokenizer("french"))
        summarizer = LsaSummarizer()
        summary = summarizer(parser.document, sentences_count)
        return " ".join(str(sentence) for sentence in summary)
    except Exception as e:
        print(f"Erreur dans le résumé : {e}")
        return "Erreur lors du résumé"

# Fonction principale
def cluster_and_generate_summary(db, documents):
    # Regrouper les articles par cluster_id
    clusters = {}
    for doc in documents:
        cluster_id = doc['cluster_id']
        content = doc['content']
        if cluster_id not in clusters:
            clusters[cluster_id] = ""
        clusters[cluster_id] += content + " "

    # Résumés parallèles
    results = Parallel(n_jobs=-1)(
        delayed(lambda cid, text: (cid, generate_summary(text)))(cid, content)
        for cid, content in clusters.items()
    )
    cluster_summaries = dict(results)

    # 🔽 Modification ici : nom du fichier de sortie
    with open('cluster_summaries_LSA.json', 'w', encoding='utf-8') as f:
        json.dump(cluster_summaries, f, ensure_ascii=False, indent=4)

    print("✅ Résumés sauvegardés dans 'cluster_summaries_LSA.json'.")

# Exécution
if __name__ == "__main__":
    print("🚀 Démarrage du script...")
    db, documents = get_data_from_mongodb()
    if documents:
        cluster_and_generate_summary(db, documents)
    else:
        print("⚠️ Aucun document trouvé.")
    print("✅ Script terminé.")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hajar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


🚀 Démarrage du script...
✅ Résumés sauvegardés dans 'cluster_summaries_LSA.json'.
✅ Script terminé.
